In [1]:
import sys
sys.path.insert(1, '../scripts/')
from preprocess import preprocess

First, format the project input files and generate the environment

In [2]:
# # downloaded project inputs
# input_data_path, build_files_path = preprocess.unpack_files(data_files = '/data2/hratch/human_me/data.zip', 
#                                         build_files = '/data2/hratch/human_me/build_files.zip', 
#                                         data_out = '/data2/hratch/human_me/raw')
# preprocess.create_environment(input_data_path, build_files_path, root_path = '/home/hratch/Projects/human_me/',
#                               processed_data_path = '/data2/hratch/human_me/processed/', 
#                               n_cores = 20)

In [3]:
from preprocess import correct_inputs 

full model

In [4]:
# correct_inputs.correct_model(model_file = '/data2/hratch/human_me/input_files/recon2_2.xml', 
#                  psim_file = '/data2/hratch/human_me/input_files/psim_recon2_2.csv')

# correct_inputs.correct_psim(psim_file = '/data2/hratch/human_me/input_files/psim_recon2_2.csv')
# # # optional - only if you want to express non-machinery proteins
# # correct_inputs.check_non_machinery(nonmachinery_file = '/data2/hratch/human_me/input_files/non_machinery.txt')

# from expression import build_me_model
# me_model, builder = build_me_model.build_me(minimal_proteome = False, compress_mrna = False)

# import pickle
# lp_path = '/data2/hratch/human_me/test_lp/'
# with open(lp_path + 'me_model.pickle', 'wb') as handle:
#     pickle.dump(me_model, handle)

toy model

In [5]:
# correct_inputs.correct_model(model_file = '/data2/hratch/human_me/input_files/toy_model.xml', 
#                  psim_file = '/data2/hratch/human_me/input_files/psim_recon2_2.csv')

# correct_inputs.correct_psim(psim_file = '/data2/hratch/human_me/input_files/psim_recon2_2.csv')

# from expression import build_me_model
# toy_me_model, builder = build_me_model.build_me(minimal_proteome = True, compress_mrna = True, 
#                                                 model_id = 'toy_me_model')

# import pickle
# lp_path = '/data2/hratch/human_me/test_lp/'
# with open(lp_path + 'toy_me_model.pickle', 'wb') as handle:
#     pickle.dump(toy_me_model, handle)

# testing

In [6]:
import h5py
import numpy as np
import pandas as pd
import pickle

In [7]:
from expression import build_me_model
toy_me_model, builder = build_me_model.build_me(minimal_proteome = True, compress_mrna = True, 
                                                model_id = 'toy_me_model')

ERROR:cobra.io.sbml:No objective coefficients in model. Unclear what should be optimized


Generate ubiquitin reactions for proteasomal degrdation
Generate ribosome


../scripts/expression/protein_expression/ubiquitin.py:34 SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
../scripts/expression/protein_expression/ubiquitin.py:65 SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
../scripts/expression/gene_information.py:114 UserWarning: HGNC:10368: The letter X is in the protein sequence. Replacing with a random amino acid
  0%|          | 2/591 [00:00<00:29, 19.67it/s]

Generate protein expression reactions for metabolic enzymes and non-machinery


100%|██████████| 591/591 [00:53<00:00, 11.09it/s]


...

A total of 1565 reactions were dropped when forming a minimal proteome
Add machinery to metabolic module reactions


  0%|          | 0/8611 [00:00<?, ?it/s]

Add machinery to expression module reactions


100%|██████████| 8611/8611 [02:35<00:00, 55.48it/s]  


Generate ME-Model
Time to build: 21.149377083778383 minutes


In [8]:
counter = 2 # 0,1 done

In [9]:
sln, stat,_ = toy_me_model.solve_lp(1e-9)

fn = '/data2/hratch/human_me/test_lp/S_matrix.h5'
if stat == 0:
    S = toy_me_model.create_stoichiometric_matrix(mu_val = 1, inplace = False, array_type = 'pandas')
    S.to_hdf(fn, key = str(counter), mode = 'a')
    print('Last saved file: {}'.format(counter))
else:
    raise ValueError('Model did not solve')

Getting MINOS parameters...
Done in 145.314 seconds with status 0


/home/hratch/anaconda3/envs/CD8T_RA/lib/python3.6/site-packages/tables/path.py:155 NaturalNameWarning: object name is not a valid Python identifier: '2'; it does not match the pattern ``^[a-zA-Z_][a-zA-Z0-9_]*$``; you will not be able to use natural naming to access this object; using ``getattr()`` will still work, though


Last saved file: 2


In [ ]:
S_1 = pd.read_hdf(fn, key = str(counter))
S_0 = pd.read_hdf(fn, key = '0')

if not S_0.equals(S_1):
    indeces = True
    if indeces:
        if S_1.shape != S_0.shape:
            print('Dimensions are not the same')
        if len(set(S_1.columns).difference(S_0.columns)) > 0:
            print('Columns are not the same')
            indeces = False
        if len(set(S_1.index).difference(S_0.index)) > 0:
            print('Rows are not the same')
            indeces = False
    if indeces:
        S_1 = S_1.loc[S_0.index, S_0.columns]
        if not S_0.equals(S_1):
            mismatch = np.argwhere(np.not_equal(S_0.values, S_1.values))
            raise ValueError('Dataframes are not equal due to stoichiometric values mismatch')
        else:
            print('Success')
            lp_path = '/data2/hratch/human_me/test_lp/'
            with open(lp_path + 'working_version.pickle', 'wb') as handle:
                pickle.dump(toy_me_model, handle)
    else:
        raise ValueError('Dataframes are not equal due to column/row label mismatch')
else:
    print('Success')
    lp_path = '/data2/hratch/human_me/test_lp/'
    with open(lp_path + 'working_version_' + str(counter) + '.pickle', 'wb') as handle:
        pickle.dump(toy_me_model, handle)

In [ ]:
# very next thing: change protein mass, it is being calculated incorrectly

In [ ]:
# row_mapper = dict(zip(sorted(set(S_1.index).difference(S_0.index)), sorted(set(S_0.index).difference(S_1.index))))
# S_1.rename(index = row_mapper,inplace = True)

# rxn_name = S_0.columns[mismatch[0][1]]
# print('rxn: ' + rxn_name)
# print('metabolite: ' + S_0.index[mismatch[0][0]])
# print('orign value: {}'.format(S_0.iloc[tuple(mismatch[0])]))
# print('new value: {}'.format(S_1.iloc[tuple(mismatch[0])]))

In [ ]:
# original
# due to a difference in complex biomass (precision issue)

# rxn: 60s_maturation
# metabolite: biomass_protein
# orign value: -1.1368683772161603e-13
# new value: 0.0